# Feature Engineering

Задача этого этапа - построить признаки для пар `session-candidate`, которые способствуют оценке релевантности товара для конкретных сессий.

В дальнейшем модель будет присваивать score каждому кандидату, вследствие чего он отсортирует объекты по этому значению. В итоге сформируется топ-20 рекомендаций.

# Построение Feature Engineering

На предыдущем этапе для каждой сессии был сформирован пул из 100 потенциально релевантных товаров. Далее нам необходимо для каждой пары `session-candidate` построить признаки, которые будут использоваться моделью для ранжирования товаров.

Первым шагом построим признаки, характеризующие взаимодействия пользователя с товаром внутри текущей сессии `history`:

- `item_click_count` — количество кликов по товару;
- `item_cart_count` — количество добавлений товара в корзину;
- `item_order_count` — количество заказов товара;
- `item_total_count` — общее количество взаимодействий с товаром.

Если случилось так, что товар был найден через co-visitation и ранее еще не встречался в текущей сесссии, то все значения этих признаков будут равняться 0.

Загружаем данные (10000 строк):

In [3]:
import pandas as pd 
from pathlib import Path

data_path = Path("../data/otto-recsys-train.jsonl")

df = pd.read_json(
    data_path, 
    lines=True,
    nrows=10000
)

df.head()

,session,events
0,0,"[{'aid': 1517085, 'ts': 1659304800025, 'type':..."
1,1,"[{'aid': 424964, 'ts': 1659304800025, 'type': ..."
2,2,"[{'aid': 763743, 'ts': 1659304800038, 'type': ..."
3,3,"[{'aid': 1425967, 'ts': 1659304800095, 'type':..."
4,4,"[{'aid': 613619, 'ts': 1659304800119, 'type': ..."


Восстанавливаем `future/history`:

In [5]:
history_arr = []
future_arr = []

for events in df['events']:
    future_size = max(1, int(len(events) * 0.25))

    history = events[:-future_size]
    future = events[-future_size:]

    history_arr.append(history)
    future_arr.append(future)

df['history'] = history_arr
df['future'] = future_arr

## Признаки взаимодействий с кандидатом

Сначала построим функцию, которая для одного товара-кандидата подсчитывает его взаимодействия внутри `history` текущей сессии.

Разделение событий на `clicks`, `carts` и `orders` позволяет сохранить более подробную информацию о взаимодействиях между событиями.

In [7]:
def get_interaction_features(history, candidate):
    click_count = 0
    cart_count = 0
    order_count = 0

    for event in history:
        if event['aid'] == candidate:
            if event['type'] == 'clicks':
                click_count += 1
            elif event['type'] == 'carts':
                cart_count += 1
            elif event['type'] == 'orders':
                order_count += 1

    total_count = click_count + cart_count + order_count

    return {'item_click_count': click_count, 'item_cart_count': cart_count, 'item_order_count': order_count, 'item_total_count': total_count}

Смотрим по кажодму `event` в `history` и если `event['aid'] == candidate`, то смотрим `event['type']`.

Проверка работы функции:

In [12]:
test_history = df['history'].iloc[0]

test_candidate = test_history[0]['aid']

features = get_interaction_features(
    test_history,
    test_candidate
)

print('Товар:', test_candidate)
print(features)

Товар: 1517085
{'item_click_count': 1, 'item_cart_count': 0, 'item_order_count': 0, 'item_total_count': 1}


## Признаки давности взаимодействия

Количество взаимодействий с товаром не учитывает, насколько недвано пользователю стал интересен товар и как долго он оставался для него приоритетным.

Поэтому добавим два важных признака:
- `item_recency` — время между последним взаимодействием с товаром и концом `history`;
- `item_interaction_span` — время между первым и последним взаимодействием с товаром.

Напишем функцию для расчет этих признаков:

In [17]:
def get_time_features(history, candidate):
    history_end = history[-1]['ts']

    candidate_timestamps = []

    for event in history:
        if event['aid'] == candidate:
            candidate_timestamps.append(event['ts'])

    if len(candidate_timestamps) == 0:
        return {'item_recency:': None,
                'item_interaction_span': 0}

    first_interaction = min(candidate_timestamps)
    last_interaction = max(candidate_timestamps)

    item_recency = history_end - last_interaction
    item_interaction_span = last_interaction - first_interaction

    return {'item_recency': item_recency,
            'item_interaction_span': item_interaction_span}

Проверяем работу функции:

In [18]:
test_history = df['history'].iloc[11]

test_candidate = test_history[0]['aid']

time_features = get_time_features(
    test_history,
    test_candidate
)

print('Товар:', test_candidate)
print(time_features)

Товар: 1110510
{'item_recency': 821941617, 'item_interaction_span': 82904081}


## Признак связи с товарами сессии

При генерации кандидатов мы использовали co-visitation, то есть товары, которые часто встречались вместе в одних сессиях. Теперь пора использовать эту информацию не только для поиска кандидатов, но и как признак для будущего ранжирования.

Добавим признак:

- `covisitation_score` — суммарная сила связи товара-кандидата с товарами из `history`.

Чем больше значение, тем сильнее кандидат связан с товарами, которыми пользователь уже интересовался.

In [21]:
from collections import defaultdict, Counter

pair_counts = defaultdict(Counter)

for history in df['history']:
    session_items = []

    for event in history:
        aid = event['aid']

        if aid not in session_items:
            session_items.append(aid)

    for i in range(len(session_items)):
        for j in range(i + 1, len(session_items)):
            item_a = session_items[i]
            item_b = session_items[j]

            pair_counts[item_a][item_b] += 1
            pair_counts[item_b][item_a] += 1


item_neighbors = {}

for item, neighbors in pair_counts.items():
    item_neighbors[item] = neighbors.most_common(20)

In [22]:
def get_covisitation_score(history, candidate, item_neighbors):
    score = 0

    history_items = set()

    for event in history:
        history_items.add(event['aid'])

    for item in history_items:
        if item not in item_neighbors:
            continue

        for neighbor, count in item_neighbors[item]:
            if neighbor == candidate:
                score += count
                break

    return score

Проверка работы функции:

In [25]:
test_history = df['history'].iloc[25]

test_candidate = item_neighbors[
    test_history[0]['aid']
][0][0]

score = get_covisitation_score(
    test_history,
    test_candidate,
    item_neighbors
)

print('Кандидат:', test_candidate)
print('Co-visitation score:', score)

Кандидат: 1052469
Co-visitation score: 2


## Глобальная статистика товара

До этого мы описывали поведение товара внутри сессии. Теперь настало время добавить признаки, которые характеризуют поведение товара среди всех доступных `history`.

Выведем и посчитаем такие признаки:

- `item_click_rate` — доля `clicks` среди взаимодействий с товаром;
- `item_cart_rate` — доля `carts` среди взаимодействий с товаром;
- `item_order_rate` — доля `orders` среди взаимодействий с товаром.

Эти признаки позволят понять, какие товары преимущественно просматривают, добавляют в корзину или заказывают.

In [26]:
item_stats = {}

for history in df['history']:
    for event in history:
        aid = event['aid']
        event_type = event['type']

        if aid not in item_stats:
            item_stats[aid] = {
                'clicks': 0,
                'carts': 0,
                'orders': 0,
                'total': 0
            }

        item_stats[aid][event_type] += 1
        item_stats[aid]['total'] += 1

In [28]:
item_rates = {}

for aid, stats in item_stats.items():
    total = stats['total']

    item_rates[aid] = {
        'item_click_rate': stats['clicks'] / total,
        'item_cart_rate': stats['carts'] / total,
        'item_order_rate': stats['orders'] / total}

Проверка:

In [32]:
test_item = next(iter(item_rates))

print('Товар:', test_item)
print('Количество событий:', item_stats[test_item])
print('Доли событий:', item_rates[test_item])

Товар: 1517085
Количество событий: {'clicks': 1, 'carts': 0, 'orders': 0, 'total': 1}
Доли событий: {'item_click_rate': 1.0, 'item_cart_rate': 0.0, 'item_order_rate': 0.0}


Полученные признаки описывают глобальный характер взаимодействий с товаром.

Например, высокий `item_order_rate` означает, что среди всех взаимодействий с данным товаром относительно большая доля приходится на заказы.

Важно: данные признаки являются долями типов событий, а не конверсиями между этапами воронки.

## Формирование обучающей таблицы

На предыдущих этапах были подготовлены признаки, описывающие товар, который является кандидатом, с разных сторон:

- взаимодействия пользователя с товаром в текущей сессии;
- давность взаимодействий;
- связь кандидата с другими товарами сессии;
- глобальное поведение товара.

Теперь объединим эти признаки в одну таблицу.

Одна строка таблицы будет соответствовать одной паре:

`session + candidate`

Для каждой пары будут рассчитаны признаки, которые можно использовать в модели ранжирования.

Сначала запишем уже известную нам функцию отбора кандидатов:

In [33]:
def generate_candidates(history, item_neighbors, k=100):
    candidates = []
    seen = set()

    for event in reversed(history):
        aid = event['aid']
        if aid not in seen:
            candidates.append(aid)
            seen.add(aid)
        if len(candidates) == k:
            return candidates
            
    history_items = list(seen)

    for aid in history_items:
        if aid not in item_neighbors:
            continue
        for neighbor, count in item_neighbors[aid]:
            if neighbor not in seen:
                candidates.append(neighbor)
                seen.add(neighbor)
            if len(candidates) == k:
                return candidates

    return candidates

Проверка работы функции:

In [35]:
test_history = df['history'].iloc[0]

test_candidates = generate_candidates(
    test_history,
    item_neighbors,
    k=100
)
print('Количество кандидатов:', len(test_candidates))
print('Первые 10 кандидатов:', test_candidates[:10])
print('Уникальных кандидатов:', len(set(test_candidates)))

Количество кандидатов: 100
Первые 10 кандидатов: [661144, 570955, 1760145, 974651, 275288, 543308, 138431, 1157411, 190818, 102416]
Уникальных кандидатов: 100


## Объединение признаков кандидата

Теперь для каждого товара-кандидата объединим рассчитанные ранее признаки.

Каждая строка будущей обучающей таблицы будет соответствовать паре `session + candidate`.

Для кандидата будем использовать:

- количество `clicks`, `carts` и `orders` внутри текущей сессии;
- общее количество взаимодействий;
- давность последнего взаимодействия;
- длительность интереса к товару;
- силу связи кандидата с товарами текущей сессии через co-visitation;
- глобальные доли `clicks`, `carts` и `orders` для товара.

Пишем функцию объединения:

In [38]:
def get_candidate_features(history, candidate, item_neighbors, item_rates):
    interaction_features = get_interaction_features(
        history, candidate
    )

    time_features = get_time_features(
        history, candidate
    )

    covisitation_score = get_covisitation_score(
        history, candidate, item_neighbors
    )

    if candidate in item_rates:
        global_features = item_rates[candidate]
    else:
        global_features = {
            'item_click_rate': 0,
            'item_cart_rate': 0,
            'item_order_rate': 0
        }

    features = {}

    features.update(interaction_features)
    features.update(time_features)
    features['covisitation_score'] = covisitation_score
    features.update(global_features)

    return features

Проверка работы функции:

In [39]:
test_candidate = test_candidates[0]

features = get_candidate_features(
    test_history,
    test_candidate,
    item_neighbors,
    item_rates
)

print('Кандидат:', test_candidate)

for feature, value in features.items():
    print(feature, ':', value)

Кандидат: 661144
item_click_count : 2
item_cart_count : 0
item_order_count : 0
item_total_count : 2
item_recency : 0
item_interaction_span : 172938592
covisitation_score : 4
item_click_rate : 1.0
item_cart_rate : 0.0
item_order_rate : 0.0


## Формирование целевой переменной

Для обучения модели ранжирования каждому товару-кандидату необходимо присвоить уровень релевантности на основе поведения пользователя в `future`.

Используем следующие уровни:

- `0` — товар не появился в `future`;
- `1` — произошёл `click`;
- `2` — товар был добавлен в `cart`;
- `3` — произошёл `order`.

Если для одного товара в `future` произошло несколько типов событий, используется наиболее значимое событие.

Таким образом модель сможет учитывать разную ценность будущих действий пользователя при ранжировании кандидатов.

In [40]:
def get_target(future, candidate):
    target = 0

    for event in future:
        if event['aid'] != candidate:
            continue

        if event['type'] == 'clicks':
            target = max(target, 1)

        elif event['type'] == 'carts':
            target = max(target, 2)

        elif event['type'] == 'orders':
            target = max(target, 3)

    return target

Проверка работы функции:

In [50]:
test_future = df['future'].iloc[0]

for candidate in test_candidates[:20]:
    target = get_target(
        test_future,
        candidate
    )

    print(candidate, '->', target)

661144 -> 0
570955 -> 0
1760145 -> 1
974651 -> 1
275288 -> 1
543308 -> 3
138431 -> 0
1157411 -> 0
190818 -> 0
102416 -> 0
1436439 -> 0
738987 -> 0
1624436 -> 0
1639229 -> 0
1521766 -> 0
280978 -> 0
1766089 -> 0
7563 -> 0
1433235 -> 0
6851 -> 0


## Формирование обучающей выборки

Теперь для каждой пары `сессия — товар-кандидат` объединим рассчитанные признаки и целевую переменную.

Одна строка будущей таблицы будет соответствовать одному товару-кандидату внутри конкретной сессии.

В таблицу войдут:

- идентификатор сессии;
- идентификатор товара-кандидата;
- признаки взаимодействия пользователя с товаром;
- временные признаки;
- `covisitation_score`;
- глобальные признаки товара;
- `target` — уровень релевантности товара в `future`.

Полученную таблицу будем использовать для обучения модели ранжирования.

In [51]:
rows = []

for _, row in df.iterrows():
    session = row['session']
    history = row['history']
    future = row['future']

    candidates = generate_candidates(
        history,
        item_neighbors,
        k=100)

    for candidate in candidates:
        features = get_candidate_features(
            history,
            candidate,
            item_neighbors,
            item_rates)

        target = get_target(
            future,
            candidate)

        features['session'] = session
        features['aid'] = candidate
        features['target'] = target

        rows.append(features)

### Итоговая таблица признаков

Преобразуем собранные строки в `DataFrame`. Каждая строка соответствует одной паре `сессия - товар-кандидат` и содержит признаки кандидата и целевую переменную.

In [54]:
train_df = pd.DataFrame(rows)

print('Размер таблицы:', train_df.shape)
train_df.head()

Размер таблицы: (744109, 14)


,item_click_count,item_cart_count,item_order_count,item_total_count,item_recency,item_interaction_span,covisitation_score,item_click_rate,item_cart_rate,item_order_rate,session,aid,target,item_recency:
0,2,0,0,2,0.0,172938592,4,1.000000,0.000000,0.000000,0,661144,0,NaN
1,3,0,0,3,14996.0,214214139,0,1.000000,0.000000,0.000000,0,570955,0,NaN
2,3,1,0,4,44215.0,172879821,4,0.833333,0.166667,0.000000,0,1760145,1,NaN
3,2,4,0,6,49336.0,214123637,0,0.600000,0.400000,0.000000,0,974651,1,NaN
4,2,1,0,3,75152.0,748586141,0,0.629630,0.222222,0.148148,0,275288,1,NaN


### Проверка итоговой выборки

Перед обучением модели проверим:

1. распределение целевой переменной;
2. наличие пропущенных значений;
3. количество кандидатов в каждой сессии.

Это позволит убедиться, что обучающая выборка сформирована вполне корректно.

In [55]:
print(train_df['target'].value_counts().sort_index())

print('\nДоли классов:')
print(train_df['target'].value_counts(normalize=True).sort_index())

target
0    723397
1     16672
2      1851
3      2189
Name: count, dtype: int64

Доли классов:
target
0    0.972165
1    0.022405
2    0.002488
3    0.002942
Name: proportion, dtype: float64


Пропуски:

In [57]:
train_df.isna().sum()

item_click_count              0
item_cart_count               0
item_order_count              0
item_total_count              0
item_recency             487758
item_interaction_span         0
covisitation_score            0
item_click_rate               0
item_cart_rate                0
item_order_rate               0
session                       0
aid                           0
target                        0
item_recency:            744109
dtype: int64

Количество кандидатов на сессюи:

In [58]:
candidates_per_session = train_df.groupby('session').size()
candidates_per_session.describe()

count    10000.000000
mean        74.410900
std         33.899194
min          1.000000
25%         43.000000
50%        100.000000
75%        100.000000
max        100.000000
dtype: float64

Для кандидатов, которых вообще не было в `history`, логично дать специально большое значение:

In [60]:
max_recency = train_df['item_recency'].max()

train_df['item_recency'] = train_df['item_recency'].fillna(
    max_recency + 1
)

In [61]:
train_df.isna().sum()

item_click_count              0
item_cart_count               0
item_order_count              0
item_total_count              0
item_recency                  0
item_interaction_span         0
covisitation_score            0
item_click_rate               0
item_cart_rate                0
item_order_rate               0
session                       0
aid                           0
target                        0
item_recency:            744109
dtype: int64

## Итог этапа 

Для каждого товара-кандидата были сформированы признаки четырёх групп:

- взаимодействия товара с пользователем внутри текущей сессии;
- временные характеристики взаимодействий;
- сила связи кандидата с товарами сессии через co-visitation;
- глобальная статистика поведения пользователей с товаром.

Была сформирована целевая переменная, которая имел диапазон релевантности товара от 0 до 3.

Итоговая обучающая выборка содержит около 745 тысяч пар `сессия — товар-кандидат`.
Пропущенные значения обработаны, а число кандидатов на одну сессию не превышает 100.

Далее приступим к обучению модели ранжирования.